# LogRAG on Kaggle: shared-protocol BGL

This notebook configures BGL LogRAG against `shared-benchmark-v1` using Ollama with the Mistral-7B-Instruct-v0.1 Q4_K_M GGUF from Hugging Face and local Hugging Face embeddings. It does not need an API key. Enable Kaggle Internet and a GPU accelerator, then attach the structured BGL CSV, BigLog model directory, and protocol-v1 artifacts as input datasets. The original `mistralai/Mistral-7B-Instruct-v0.1` repository is Safetensors; Ollama needs the GGUF conversion configured below. HDFS remains separate. The benchmark cell is disabled until you explicitly set `RUN_FULL_BENCHMARK = True`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/code-papers-doing-copilot')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/rubyhcm/code-papers-doing-copilot.git',
        str(REPO_DIR),
    ], check=True)
print('Repository:', REPO_DIR)

In [ ]:
# Retain Kaggle's preinstalled CUDA-enabled torch/torchvision.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-r',
    str(REPO_DIR / 'LogRAG' / 'requirements-kaggle.txt'),
], check=True)

In [ ]:
import json
import os
import shutil
import subprocess
import time
import requests

OLLAMA_MODEL = 'hf.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF:Q4_K_M'
OLLAMA_BASE_URL = 'http://127.0.0.1:11434'
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_BASE_URL'] = OLLAMA_BASE_URL

if shutil.which('ollama') is None:
    if shutil.which('zstd') is None:
        if shutil.which('apt-get') is None:
            raise RuntimeError('Ollama installer needs zstd; apt-get is unavailable.')
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
    subprocess.run(
        ['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True,
    )

OLLAMA_LOG = open('/kaggle/working/ollama.log', 'w', encoding='utf-8')
OLLAMA_PROCESS = None
try:
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=2).raise_for_status()
except requests.RequestException:
    OLLAMA_PROCESS = subprocess.Popen(
        ['ollama', 'serve'],
        env=os.environ.copy(),
        stdout=OLLAMA_LOG,
        stderr=subprocess.STDOUT,
    )
    for attempt in range(60):
        if OLLAMA_PROCESS.poll() is not None:
            raise RuntimeError('Ollama server exited; inspect /kaggle/working/ollama.log')
        try:
            requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=2).raise_for_status()
            break
        except requests.RequestException:
            if attempt == 59:
                raise TimeoutError('Ollama server did not become ready; inspect ollama.log')
            time.sleep(2)

subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
smoke_response = requests.post(
    f'{OLLAMA_BASE_URL}/api/chat',
    json={
        'model': OLLAMA_MODEL,
        'messages': [{'role': 'user', 'content': 'Return a JSON object with key ready and value true.'}],
        'stream': False,
        'keep_alive': 0,
        'options': {'temperature': 0, 'num_predict': 64},
    },
    timeout=300,
)
if not smoke_response.ok:
    raise RuntimeError(
        f'Ollama smoke test failed ({smoke_response.status_code}): '
        f'{smoke_response.text}'
    )
json.loads(smoke_response.json()['message']['content'])
print('Ollama model ready:', OLLAMA_MODEL)

In [ ]:
import os
from datetime import datetime
import torch

# Replace these three paths with the attached Kaggle input locations.
BGL_STRUCTURED_CSV = Path('/kaggle/input/REPLACE/BGL.log_structured.csv')
BIGLOG_ENCODER = Path('/kaggle/input/REPLACE/biglog')
PROTOCOL_DIR = Path('/kaggle/input/REPLACE/benchmark_protocol_v1')

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
assert BGL_STRUCTURED_CSV.is_file(), BGL_STRUCTURED_CSV
assert (BIGLOG_ENCODER / 'config.json').is_file(), BIGLOG_ENCODER
assert (PROTOCOL_DIR / 'bgl.csv').is_file(), PROTOCOL_DIR
assert (PROTOCOL_DIR / 'manifest.json').is_file(), PROTOCOL_DIR

os.environ['LOGRAG_CHAT_MODEL'] = OLLAMA_MODEL
os.environ['LOGRAG_LLM_BACKEND'] = 'ollama'
os.environ['LOGRAG_EMBEDDING_BACKEND'] = 'huggingface'
os.environ['LOGRAG_EMBEDDING_MODEL'] = 'sentence-transformers/all-MiniLM-L6-v2'
os.environ['LOGRAG_EMBEDDING_DEVICE'] = 'cpu'
os.environ['LOGRAG_OLLAMA_NUM_CTX'] = '4096'
os.environ['LOGRAG_OLLAMA_NUM_PREDICT'] = '512'
os.environ['LOGRAG_STRUCTURED_CSV'] = str(BGL_STRUCTURED_CSV)
os.environ['LOGRAG_ENCODER_PATH'] = str(BIGLOG_ENCODER)
os.environ['LOGRAG_PROTOCOL_DIR'] = str(PROTOCOL_DIR)
RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
os.environ['LOGRAG_SHARED_OUTPUT_DIR'] = f'/kaggle/working/lograg-results-{RUN_ID}'
os.environ['LOGRAG_OUTPUT_DIR'] = os.environ['LOGRAG_SHARED_OUTPUT_DIR']
os.environ['LOGRAG_PERSIST_DIRECTORY'] = f'/kaggle/working/lograg-results-{RUN_ID}/ragdb-bgl'
os.environ['LOGRAG_SEED'] = '42'
os.environ['LOGRAG_EPOCHS'] = '50'
os.environ['LOGRAG_BATCH_SIZE'] = '40960'
os.environ['LOGRAG_EMBEDDING_BATCH_SIZE'] = '32'

print('GPU:', torch.cuda.get_device_name(0))
print('Local Ollama chat and Hugging Face embeddings configured; no API key is used.')

In [ ]:
import os
os.chdir(REPO_DIR / 'LogRAG')
sys.path.insert(0, str(REPO_DIR / 'LogRAG'))
from shared_benchmark import prepare_bgl_splits

SPLIT_DIR = Path(os.environ['LOGRAG_SHARED_OUTPUT_DIR']) / 'input_splits'
split_paths = prepare_bgl_splits(BGL_STRUCTURED_CSV, PROTOCOL_DIR, SPLIT_DIR)
for split, path in split_paths.items():
    with open(path, encoding='utf-8') as stream:
        record_count = sum(1 for _ in stream) - 1
    print(split, 'records:', record_count)

The preflight above checks row order, required columns, exact 200-record windows, labels, and record counts against the manifest. Ollama uses the Q4_K_M GGUF; the separate sentence-transformers embedding model is downloaded from Hugging Face and runs on CPU. Review available GPU/disk space before enabling the full training cell. This setup uses the plan's 50-epoch profile; document that configuration in the thesis.

In [ ]:
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    subprocess.run([sys.executable, 'main_shared.py'], check=True)
else:
    print('Preflight passed. Set RUN_FULL_BENCHMARK=True to train and run local Ollama RAG.')

In [ ]:
RESULTS_DIR = Path(os.environ['LOGRAG_SHARED_OUTPUT_DIR'])
if (RESULTS_DIR / 'results.json').exists():
    print((RESULTS_DIR / 'results.json').read_text(encoding='utf-8'))
    print('Download the lograg-results output directory from the Kaggle notebook session.')
else:
    print('No benchmark results yet; run the full benchmark cell first.')